## Embeddings

No nosso exemplo anterior, operamos com vetores de saco de palavras de alta dimensionalidade com comprimento `vocab_size`, e estávamos explicitamente convertendo de vetores de representação posicional de baixa dimensão para representação esparsa one-hot. Essa representação one-hot não é eficiente em memória; além disso, cada palavra é tratada independentemente das outras, ou seja, vetores codificados one-hot não expressam qualquer similaridade semântica entre as palavras.

Nesta unidade, continuaremos explorando o conjunto de dados **News AG**. Para começar, vamos carregar os dados e obter algumas definições do notebook anterior.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## O que é embedding?

A ideia de **embedding** é representar palavras por vetores densos de baixa dimensionalidade, que de alguma forma refletem o significado semântico de uma palavra. Mais tarde discutiremos como construir embeddings de palavras significativos, mas por enquanto vamos apenas pensar em embeddings como uma forma de reduzir a dimensionalidade de um vetor de palavra.

Então, a camada de embedding receberia uma palavra como entrada e produziria um vetor de saída da dimensão especificada `embedding_size`. Em certo sentido, é muito semelhante à camada `Linear`, mas em vez de receber um vetor codificado em one-hot, será capaz de receber um número de palavra como entrada.

Ao usar a camada de embedding como primeira camada em nossa rede, podemos mudar do modelo bag-of-words para o modelo **embedding bag**, onde primeiro convertimos cada palavra do nosso texto em seu embedding correspondente e depois calculamos alguma função agregada sobre todos esses embeddings, como `sum`, `average` ou `max`.

![Imagem mostrando um classificador embedding para cinco palavras de sequência.](../../../../../translated_images/pt-BR/embedding-classifier-example.b77f021a7ee67eee.webp)

Nossa rede neural classificador começará com a camada de embedding, depois a camada de agregação e um classificador linear no topo disso:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Lidando com tamanho variável de sequência

Como resultado dessa arquitetura, os minibatches para nossa rede precisariam ser criados de uma certa maneira. Na unidade anterior, ao usar bag-of-words, todos os tensores BoW em um minibatch tinham tamanho igual `vocab_size`, independentemente do comprimento real da nossa sequência de texto. Uma vez que passamos para word embeddings, acabaríamos com um número variável de palavras em cada amostra de texto, e ao combinar essas amostras em minibatches teríamos que aplicar algum padding.

Isso pode ser feito usando a mesma técnica de fornecer a função `collate_fn` para a fonte de dados:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Treinando classificador de embedding

Agora que definimos o dataloader apropriado, podemos treinar o modelo usando a função de treinamento que definimos na unidade anterior:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Nota**: Estamos treinando apenas para 25 mil registros aqui (menos de uma época completa) para economizar tempo, mas você pode continuar treinando, escrever uma função para treinar por várias épocas e experimentar o parâmetro de taxa de aprendizado para alcançar maior precisão. Você deverá conseguir obter uma precisão de cerca de 90%.


### Camada EmbeddingBag e Representação de Sequência de Comprimento Variável

Na arquitetura anterior, precisávamos preencher todas as sequências para o mesmo comprimento a fim de encaixá-las em um minibatch. Esta não é a forma mais eficiente de representar sequências de comprimento variável - outra abordagem seria usar um vetor de **offset**, que conteria os deslocamentos de todas as sequências armazenadas em um vetor grande.

![Imagem mostrando uma representação de sequência por offset](../../../../../translated_images/pt-BR/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Nota**: Na imagem acima, mostramos uma sequência de caracteres, mas em nosso exemplo estamos trabalhando com sequências de palavras. Entretanto, o princípio geral de representar sequências com vetor de offset permanece o mesmo.

Para trabalhar com a representação por offset, usamos a camada [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Ela é similar ao `Embedding`, mas recebe um vetor de conteúdo e um vetor de offset como entrada, e também inclui uma camada de média, que pode ser `mean`, `sum` ou `max`.

Aqui está a rede modificada que usa `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Para preparar o conjunto de dados para treinamento, precisamos fornecer uma função de conversão que preparará o vetor de deslocamento:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Observe que, diferente de todos os exemplos anteriores, nossa rede agora aceita dois parâmetros: vetor de dados e vetor de offset, que possuem tamanhos diferentes. Da mesma forma, nosso carregador de dados também nos fornece 3 valores em vez de 2: tanto os vetores de texto quanto os de offset são fornecidos como características. Portanto, precisamos ajustar ligeiramente nossa função de treinamento para lidar com isso:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Embeddings Semânticos: Word2Vec

No nosso exemplo anterior, a camada de embedding do modelo aprendeu a mapear palavras para uma representação vetorial, no entanto, essa representação não tinha muito significado semântico. Seria interessante aprender uma representação vetorial onde palavras semelhantes ou sinônimos corresponderiam a vetores próximos entre si em termos de alguma distância vetorial (ex: distância euclidiana).

Para isso, precisamos pré-treinar nosso modelo de embedding em uma grande coleção de textos de uma forma específica. Uma das primeiras maneiras de treinar embeddings semânticos é chamada [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Ela é baseada em duas arquiteturas principais que são usadas para produzir uma representação distribuída das palavras:

 - **Continuous bag-of-words** (CBoW) — nesta arquitetura, treinamos o modelo para prever uma palavra a partir do contexto ao redor. Dado o ngrama $(W_{-2},W_{-1},W_0,W_1,W_2)$, o objetivo do modelo é prever $W_0$ a partir de $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** é o oposto do CBoW. O modelo usa a janela de contexto ao redor para prever a palavra atual.

O CBoW é mais rápido, enquanto o skip-gram é mais lento, mas faz um trabalho melhor em representar palavras pouco frequentes.

![Imagem mostrando os algoritmos CBoW e Skip-Gram para converter palavras em vetores.](../../../../../translated_images/pt-BR/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Para experimentar com o embedding word2vec pré-treinado no conjunto de dados Google News, podemos usar a biblioteca **gensim**. Abaixo encontramos as palavras mais similares a 'neural'

> **Nota:** Quando você cria vetores de palavras pela primeira vez, pode levar algum tempo para baixá-los!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Também podemos calcular embeddings vetoriais da palavra, para serem usados no treinamento do modelo de classificação (mostramos apenas os primeiros 20 componentes do vetor para maior clareza):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

A grande vantagem das embeddings semânticas é que você pode manipular a codificação vetorial para mudar a semântica. Por exemplo, podemos pedir para encontrar uma palavra cuja representação vetorial seja tão próxima quanto possível das palavras *king* e *woman*, e tão distante quanto possível da palavra *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Tanto CBoW quanto Skip-Grams são embeddings “previsivos”, no sentido de que consideram apenas contextos locais. O Word2Vec não aproveita o contexto global. 

**FastText** expande o Word2Vec aprendendo representações vetoriais para cada palavra e os n-gramas de caracteres encontrados em cada palavra. Os valores das representações são então promediados em um único vetor a cada passo de treinamento. Embora isso adicione muito mais cálculo ao pré-treinamento, permite que os embeddings de palavras codifiquem informações sub-palavra. 

Outro método, **GloVe**, aproveita a ideia da matriz de co-ocorrência, utiliza métodos neurais para decompor a matriz de co-ocorrência em vetores de palavras mais expressivos e não lineares.

Você pode experimentar o exemplo alterando os embeddings para FastText e GloVe, já que o gensim suporta vários modelos diferentes de embeddings de palavras.


## Usando Embeddings Pré-Treinados em PyTorch

Podemos modificar o exemplo acima para pré-popular a matriz em nossa camada de embedding com embeddings semânticos, como Word2Vec. Precisamos levar em conta que os vocabulários do embedding pré-treinado e do nosso corpus de texto provavelmente não coincidirão, então inicializaremos os pesos para as palavras ausentes com valores aleatórios:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Agora vamos treinar nosso modelo. Observe que o tempo necessário para treinar o modelo é significativamente maior do que no exemplo anterior, devido ao tamanho maior da camada de embedding e, portanto, ao número muito maior de parâmetros. Além disso, por causa disso, talvez precisemos treinar nosso modelo com mais exemplos se quisermos evitar overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

No nosso caso, não vemos um grande aumento na precisão, o que provavelmente se deve a vocabulários bastante diferentes. 
Para superar o problema de vocabulários diferentes, podemos usar uma das seguintes soluções:
* Re-treinar o modelo word2vec no nosso vocabulário
* Carregar nosso conjunto de dados com o vocabulário do modelo word2vec pré-treinado. O vocabulário usado para carregar o conjunto de dados pode ser especificado durante o carregamento.

A última abordagem parece mais fácil, especialmente porque o framework PyTorch `torchtext` contém suporte embutido para embeddings. Podemos, por exemplo, instanciar um vocabulário baseado em GloVe da seguinte maneira:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


O vocabulário carregado tem as seguintes operações básicas:
* `vocab.stoi` dicionário permite converter uma palavra em seu índice no dicionário
* `vocab.itos` faz o contrário - converte número em palavra
* `vocab.vectors` é o array de vetores de embedding, então para obter o embedding de uma palavra `s` precisamos usar `vocab.vectors[vocab.stoi[s]]`

Aqui está um exemplo de manipulação de embeddings para demonstrar a equação **kind-man+woman = queen** (eu tive que ajustar um pouco o coeficiente para funcionar):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Para treinar o classificador usando essas embeddings, primeiro precisamos codificar nosso conjunto de dados usando o vocabulário GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Como vimos acima, todos os embeddings vetoriais são armazenados na matriz `vocab.vectors`. Isso facilita muito carregar esses pesos nos pesos da camada de embedding usando uma simples cópia:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Agora vamos treinar nosso modelo e ver se obtemos melhores resultados:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Uma das razões pelas quais não estamos vendo um aumento significativo na precisão é devido ao fato de que algumas palavras do nosso conjunto de dados estão ausentes no vocabulário pré-treinado do GloVe e, portanto, são essencialmente ignoradas. Para superar esse fato, podemos treinar nossas próprias embeddings em nosso conjunto de dados.


## Embeddings Contextuais

Uma limitação chave das representações tradicionais de embeddings pré-treinadas, como Word2Vec, é o problema da desambiguação de sentido das palavras. Embora os embeddings pré-treinados possam capturar parte do significado das palavras no contexto, todos os sentidos possíveis de uma palavra são codificados no mesmo embedding. Isso pode causar problemas em modelos posteriores, pois muitas palavras, como a palavra 'play', têm significados diferentes dependendo do contexto em que são usadas.

Por exemplo, a palavra 'play' nessas duas frases diferentes têm significados bastante distintos:
- I went to a **play** at the theature.
- John wants to **play** with his friends.

Os embeddings pré-treinados acima representam ambos os significados da palavra 'play' no mesmo embedding. Para superar essa limitação, precisamos construir embeddings baseados no **modelo de linguagem**, que é treinado em um grande corpus de texto e *sabe* como as palavras podem ser combinadas em diferentes contextos. Discutir embeddings contextuais está fora do escopo deste tutorial, mas voltaremos a eles ao falarmos sobre modelos de linguagem na próxima unidade.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
